In [12]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Wed" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "Wed"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "Mon" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : d:\hanwha-agent


In [13]:
from app.models.document import Chunk
from app.db.session import session_scope
from app.models.document import Chunk
from sqlalchemy import func, select

queries = ['숙박', '숙박비', '출장 갈 때 숙박비 얼마?', '출장 숙박비']
zeros = []
with session_scope() as s:
    for q in queries:
        n = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.text.like(f'%{q}%'))) or 0
        print(f'{q!r:<28} → {n}줄')
        if n == 0:
            zeros.append(q)
print()
print(f'넷 중 {len(zeros)}개가 0 줄입니다.')

'숙박'                         → 3줄
'숙박비'                        → 3줄
'출장 갈 때 숙박비 얼마?'             → 0줄
'출장 숙박비'                     → 0줄

넷 중 2개가 0 줄입니다.


In [14]:
from sqlalchemy import text
from app.db.session import session_scope
with session_scope() as s:
    v = s.execute(text("SELECT extversion FROM pg_extension WHERE extname='vector'")).scalar()
    if v:
        print(f'vector 확장 : 이미 켜져 있습니다 — {v}')
    else:
        s.execute(text('CREATE EXTENSION IF NOT EXISTS vector'))
        s.commit()
        v = s.execute(text("SELECT extversion FROM pg_extension WHERE extname='vector'")).scalar()
        print(f'vector 확장 : 켰습니다 — {v}')

vector 확장 : 이미 켜져 있습니다 — 0.8.6


In [15]:
import importlib
from pathlib import Path
import app.models.base
importlib.reload(app.models.base)

from app.core.config import get_settings
get_settings.cache_clear()
settings = get_settings()

print('models/base.py 의 EMBED_DIM :', app.models.base.EMBED_DIM)
print('settings.embed_dim          :', settings.embed_dim)
print('settings.embed_provider     :', settings.embed_provider)
print('settings.embed_model_dir    :', settings.embed_model_dir)
print('settings.upstage_embed_model:', settings.upstage_embed_model)
print('.env :', f'{len(Path('.env').read_text(encoding='utf-8').splitlines())}줄')

models/base.py 의 EMBED_DIM : 1024
settings.embed_dim          : 1024
settings.embed_provider     : local
settings.embed_model_dir    : models/bge-m3
settings.upstage_embed_model: 
.env : 53줄


In [16]:
import importlib
import inspect
from app.services import document_service

importlib.reload(document_service)

job_src = inspect.getsource(document_service.run_ingest_job)
for needle in ['range(5)', 'steps[4]["time"]', 'int(5 / len(steps)']:
    print(f'{needle:<20}: ' + ('있음' if needle in job_src else '없음'))
path = next(SAMPLES.glob('DOC-FI-009_*_v1.4.docx')).as_posix()
job_id = document_service.start_ingest_job(doc_id='DOC-FI-009', version='v1.4', path=path)   # 파일과 같은 판(v1.4)
document_service.run_ingest_job(job_id)
job = document_service.get_job(job_id)
print()
print(f'status={job['status']} progress={job['progress']} chunk_count={job['chunk_count']}')
if job['status'] == '실패':
    print('message:', job['message'])
for step in job['steps']:
    print(f'  {step['state']:<5} {step['name']}')

range(5)            : 있음
steps[4]["time"]    : 없음
int(5 / len(steps)  : 없음


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

15:47:13 INFO     app.rag.store: 청크 적재: DOC-FI-009v1.4 - 19청크 - 19벡터

status=완료 progress=83 chunk_count=19
  ok    파일 검증
  ok    문서 파싱
  ok    표→마크다운 변환
  ok    청킹
  ok    임베딩
  todo  색인


In [17]:
from sqlalchemy import select
from app.db.session import session_scope
from app.models.document import Chunk
with session_scope() as s:
    base = s.execute(select(Chunk).where(Chunk.embedding.is_not(None)).order_by(Chunk.id).limit(1)).scalar_one()
    base_vec = list(base.embedding)
    print(f'기준 청크 : {base.locator}   (벡터 길이 {len(base_vec)})')
    OPS = {'<=>  코사인 거리': Chunk.embedding.cosine_distance, '<->  L2 거리': Chunk.embedding.l2_distance, '<#>  내적의 음수': Chunk.embedding.max_inner_product}
    for name, op in OPS.items():
        print(f'\n{name}')
        stmt = select(Chunk.locator, op(base_vec).label('d')).where(Chunk.embedding.is_not(None)).order_by('d').limit(5)
        for rank, (loc, d) in enumerate(s.execute(stmt), 1):
            print(f'  {rank}. {loc:<26} {round(float(d), 4)}')


기준 청크 : p.1 (1/5)   (벡터 길이 1024)

<=>  코사인 거리
  1. p.1 (1/5)                  0.0
  2. p.1 (3/5)                  0.2397
  3. p.1 (4/5)                  0.2667
  4. p.1 (2/5)                  0.3033
  5. p.1 (5/5)                  0.309

<->  L2 거리
  1. p.1 (1/5)                  0.0
  2. p.1 (3/5)                  0.6924
  3. p.1 (4/5)                  0.7303
  4. p.1 (2/5)                  0.7788
  5. p.1 (5/5)                  0.7862

<#>  내적의 음수
  1. p.1 (1/5)                  -1.0
  2. p.1 (3/5)                  -0.7603
  3. p.1 (4/5)                  -0.7333
  4. p.1 (2/5)                  -0.6967
  5. p.1 (5/5)                  -0.691


In [18]:
with session_scope() as s:
    stmt = select(Chunk.locator, Chunk.embedding.cosine_distance(base_vec).label('d')).where(Chunk.embedding.is_not(None)).order_by('d').limit(5)
    rows = [(loc, round(float(d), 4), round(1.0 - float(d), 4)) for loc, d in s.execute(stmt)]
by_distance = rows
by_score = sorted(rows, key=lambda r: r[2], reverse=True)
print('[ distance 오름차순 ]        [ score 내림차순 ]')
for a, b in zip(by_distance, by_score):
    print(f'  {a[1]:<8}{a[0]:<22} {b[2]:<8}{b[0]}')
print('\n두 표의 순서가 같은가 :', [r[0] for r in by_distance] == [r[0] for r in by_score])

[ distance 오름차순 ]        [ score 내림차순 ]
  0.0     p.1 (1/5)              1.0     p.1 (1/5)
  0.2397  p.1 (3/5)              0.7603  p.1 (3/5)
  0.2667  p.1 (4/5)              0.7333  p.1 (4/5)
  0.3033  p.1 (2/5)              0.6967  p.1 (2/5)
  0.309   p.1 (5/5)              0.691   p.1 (5/5)

두 표의 순서가 같은가 : True


In [20]:
import importlib
from app.rag import embedder, store
importlib.reload(store)     # 수정된 store.py 다시 불러오기
import inspect; print("'locator'" in inspect.getsource(store._row))

QUESTION='출장갈 때 숙박비 얼마?'

vec=embedder.embed_query(QUESTION)
print('질문:', QUESTION)
print('질문 벡터:', len(vec))

with session_scope() as s:
    rows=store.vector_search(s, vec, limit=3)
print()

for i, r in enumerate(rows, 1):
    print(f'{i}.{r['doc_id']}{r['version']} - {r['locator']} score{r['score']}')
    print(' ', r['text'][:40].replace('\n', ' '), '...')

print(f'벡터로 찾은 건수:{len(rows)}건')

True


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

질문: 출장갈 때 숙박비 얼마?
질문 벡터: 1024

1.DOC-FI-009v1.4 - 제2장 제11조(출장 중 사용) score0.5554
  제11조(출장 중 사용) ① 출장 중 지출은 법인카드 사용을 원칙으로 한 ...
2.DOC-FI-009v1.4 - 표2 score0.5379
  | 구분 | 항목 | 사용 가능 | 증빙 추가 사항 | |---|---| ...
3.DOC-FI-009v1.4 - 제2장 제9조(사용 한도) score0.4584
  제9조(사용 한도) ① 1회 결제 한도는 1,000,000원으로 한다.  ...
벡터로 찾은 건수:3건
